# Use the speed calculator prototype with OTAnalytics

> Note: The base path is set in `.vscode/settings.json` to workspace path. Otherwise Jupyter can't import OTAnalytics.

## Import the prototype modules

Before working with events, you need to import the event_processor to import and convert the events to a shaped data frame.

In [19]:
# Import libraries and modules
# OTAnalytics modules
from OTAnalytics.plugin_prototypes.event_parser.event_parser import EventParser
from OTAnalytics.plugin_prototypes.speed_calculator.speed_calculator import (
    SpeedCalculator,
)
from OTAnalytics.plugin_prototypes.counter.counter import Counter
import pandas as pd
import plotly.express as px


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Set config variables

In the current version, configs for prototypes are stored in dictionaries within the Jupyter notebook. In a future version, the will likely be stored in a config file.

The path to the eventlists can be a path to a single event list or a folder (all otevent files within this folder will be imported).

At each section, the counts will be calculated by direction using the flows. Therefore, you have to name the directions (default: "in" and "out"). Further, you have to provide an otflow file at the key `SECTIONSLIST_PATH`.

In [20]:
# Set env parameters and path
CONFIG = {
    "TIME_FORMAT": "%d.%m%.%y %H:%M Uhr",
    "FILTER_CLASS": [],
    "FILTER_SECTION": [],
    "EVENTLIST_PATH": "/Volumes/platomo data/Projekte/016 Videoauswertungen Esslingen MC/LAP-Maßnahme/Videos/S6/Ergebnisse/events_new.otevents",
    "SECTIONSLIST_PATH": "/Volumes/platomo data/Projekte/016 Videoauswertungen Esslingen MC/LAP-Maßnahme/Videos/S6/Ergebnisse/flows_new.otflow",
    "FROM_TIME": "2023-07-20 00:00:00",
    "TO_TIME": "2023-07-20 23:59:59",
    "INTERVAL_LENGTH_MIN": 15,  # in minutes
    "DIRECTION_NAMES": {                # Ist ein Relikt, das noch nicht entfernt wurde. Die Richtungen kommen mittlerweile aus den flows.
        "first_to_last_section": "in",
        "last_to_first_section": "out",
    },
    "SPEED_TABLE_PATH": "/Volumes/platomo data/Projekte/016 Videoauswertungen Esslingen MC/LAP-Maßnahme/Videos/S6/Ergebnisse/speeds.csv",
    "COUNT_TABLE_PATH": "/Volumes/platomo data/Projekte/016 Videoauswertungen Esslingen MC/LAP-Maßnahme/Videos/S6/Ergebnisse/counts_prototype.csv"
}

## Define modes

Here, one can specify which modes should be analysed. For every class of OTVision, the mode for analysis can be specified.

In [21]:
mode_mapper = {
    "bicyclist": "RAD",
    "bicyclist_with_trailer": "RAD",
    "cargobike_driver": "RAD",
    "car": "PKW & LFW",
    "car_with_trailer": "PKW & LFW",
    "bus": "BUS",
    "motorcyclist": "KRAD",
    "delivery_van": "PKW & LFW",
    "delivery_van_with_trailer": "PKW & LFW",
    "private_van": "PKW & LFW",
    "private_van_with_trailer": "PKW & LFW",
    "truck": "LKW ab 3,5t",
    "truck_with_trailer": "LZ/SFZ",
    "truck_with_semitrailer": "LZ/SFZ",
}

## Import Events

In [22]:
event_processor = EventParser(CONFIG)
events = event_processor.process_events()

## Calculate and Plot Speeds

In [23]:
speed_calculator = SpeedCalculator(
    CONFIG, events
)

speed_table = speed_calculator.calculate_speeds()
speed_table["road_user_type"]=speed_table["road_user_type"].map(mode_mapper)
fig = px.box(
        speed_table,
        y="speed",
        x="road_user_type",
        color="from_section",
        range_y=[0,100]
    )

fig.update_layout(
            title=f"Geschwindigkeiten",
            xaxis_title="Geschwindigkeit [km/h]",
            legend_title="From<br>section"
        )
        
fig.show()

/Users/michaelheilig/GIT/OTC/OTAnalytics/OTAnalytics/plugin_prototypes/speed_calculator/speed_calculator.py:114: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



## Create Flow Table

Here, a flow table will be calculated. This is an independent calculation from the one implemented in the OTAnalytics GUI. However, the aggregated modes given in the mode mapper are exported.

In [24]:
flow_processor = Counter(CONFIG, events)

filter_sections = []
filter_classes = []

flow_table = flow_processor.create_flow_table(filter_sections, filter_classes, pd.Timedelta("0h"))
flow_table = flow_table[flow_table["from_section"] != flow_table["to_section"]]
flow_table = flow_processor.convert_flow_table(
    flow_table, mode_mapper, aggregated=True
)

## Export tables

In [25]:
speed_table.drop(["road_user_id", "occurrence_from", "occurrence_to", "distance"], axis=1).to_csv(CONFIG["SPEED_TABLE_PATH"])
flow_table.to_csv(CONFIG["COUNT_TABLE_PATH"])